# OpenRadioss2PhysicsNeMo Dataset Generation

This notebook automates sequential OpenRadioss simulations to generate training datasets for PhysicsNeMo machine learning models. By following these steps, you will install all prerequisites needed to create your own dataset and run the PhysicsNeMo example.

Specifically, this repository provides a full pipeline for generating crash simulation data using OpenRadioss. The generated datasets are formatted to train AI models within the [PhysicsNeMo framework](https://github.com/NVIDIA/physicsnemo/tree/main/examples/structural_mechanics/crash), using the [OpenRadioss Bumper Beam example](http://openradioss.atlassian.net/wiki/spaces/OPENRADIOSS/pages/11075585/Bumper+Beam) as a baseline. Raw data is generated by iteratively modifying the shell thickness and pole position across successive simulation runs.

The OpenRadioss ANIM output files are then converted to d3plot format using the [Vortex-Radioss](https://github.com/Vortex-CAE/Vortex-Radioss) tool, enabling compatibility with the d3plot reader available in the PhysicsNeMo crash example Curator.

**To get started**, download OpenRadioss from the [official releases page](https://github.com/OpenRadioss/OpenRadioss/releases). If you only need the pre-generated dataset for training, it is available on [Hugging Face](https://huggingface.co/datasets/AIRBORNEPANDA/BumperBeamCrashExample).


## Step 1 — Download and Set Up OpenRadioss

Download the pre-built OpenRadioss Windows binaries from the official GitHub releases and extract them into the current working directory. These binaries include the **Starter** (pre-processor) and **Engine** (solver) executables required to run crash simulations.


In [ ]:
# Download OpenRadioss Windows binaries
!curl -L -o OpenRadioss_win64.zip https://github.com/OpenRadioss/OpenRadioss/releases/download/latest-20260319/OpenRadioss_win64.zip

# Extract OpenRadioss
!tar -xf OpenRadioss_win64.zip

print("OpenRadioss downloaded and extracted successfully!")

## Step 2 — Install Vortex-Radioss Dependencies

Install the Python dependencies required to convert OpenRadioss ANIM output files into the d3plot format expected by PhysicsNeMo.

- **lasso-python**: provides the d3plot read/write interface used internally by Vortex-Radioss.
- **tqdm**: displays progress bars during batch conversion.


In [ ]:
!pip install lasso-python
!pip install tqdm

In [ ]:
import sys

# Clone Vortex-Radioss repository

import sys
import shutil
from pathlib import Path
import subprocess

base_dir = Path.cwd()
repo_root = base_dir / "Vortex-Radioss"

# Clone repo
subprocess.run([
    "git", "clone",
    "https://github.com/Vortex-CAE/Vortex-Radioss",
    str(repo_root)
], check=True)

# Add repo to path
sys.path.insert(0, str(repo_root.resolve()))
shutil.move('./Vortex-Radioss/vortex_radioss', './')
try:
    from vortex_radioss.animtod3plot.Anim_to_D3plot import readAndConvert
    print("Vortex-Radioss imported successfully!")
except ImportError as e:
    print(f"Import error: {e}")
    print("Attempting alternative import path...")
    sys.path.insert(0, '/Vortex-Radioss/vortex_radioss')
    from animtod3plot.Anim_to_D3plot import readAndConvert
    print("Vortex-Radioss imported successfully via alternative path!")

## Step 3 — Create the OpenRadioss Run Script

Generate a Windows batch file (`runradioss.bat`) that configures the required environment variables and launches the OpenRadioss Starter and Engine executables for a given simulation.

The script accepts four arguments:
| Argument | Description |
|---|---|
| `%1` | Path to the OpenRadioss installation directory |
| `%2` | Path to the simulation folder |
| `%3` | Simulation file name (without extension suffix) |
| `%4` | Number of CPU threads to use |

> **Reference:** Environment variable setup follows the official [OpenRadioss installation guide](https://github.com/OpenRadioss/OpenRadioss/blob/main/INSTALL.md).


In [ ]:
# Create Windows .bat run script
runradioss_bat_content = '''
set OPENRADIOSS_PATH=%1
set sim_path=%2
set sim_file_name=%3
set number_of_threads=%4
set RAD_CFG_PATH=%OPENRADIOSS_PATH%\hm_cfg_files
set RAD_H3D_PATH=%OPENRADIOSS_PATH%\extlib\h3d\lib\win64
set PATH=%OPENRADIOSS_PATH%\extlib\hm_reader\win64;%PATH%
set PATH=%OPENRADIOSS_PATH%\extlib\intelOneAPI_runtime\win64;%PATH%
set KMP_STACKSIZE=400m
cd %sim_path%
%OPENRADIOSS_PATH%\exec\starter_win64.exe -i %sim_path%\%sim_file_name%_0000.rad -nt %number_of_threads% -np 1
%OPENRADIOSS_PATH%\exec\engine_win64_sp.exe -i %sim_path%\%sim_file_name%_0001.rad -nt %number_of_threads% 
'''

# Write the script
with open('./runradioss.bat', 'w') as f:
    f.write(runradioss_bat_content)


print(".bat run script created at ./runradioss.bat")

## Step 4 — Download the Bumper Beam Input Files

Clone the Radioss input deck for the Bumper Beam model from Hugging Face. The original [OpenRadioss Bumper Beam example](http://openradioss.atlassian.net/wiki/spaces/OPENRADIOSS/pages/11075585/Bumper+Beam) outputs results in H3D format. This repository includes a **modified engine file** configured to output ANIM files instead, which are required for the ANIM-to-d3plot conversion step.


In [ ]:
# Download the Radioss input for the Bumper Beam example
!git clone https://huggingface.co/datasets/AIRBORNEPANDA/Bumper_Beam_Radioss_Input

## Step 5 — Main Simulation Script

This section defines the core simulation loop. It covers three tasks:
1. **Detecting available CPU threads** to maximize solver performance.
2. **Defining helper functions** to modify OpenRadioss input files and launch simulations.
3. **Running the batch simulations** with randomized parameters.


### 5.1 — Detect Available CPU Threads

Query the system for the total number of logical CPU cores (threads). The solver will use all available threads by default. To limit CPU usage, manually set the `logical` variable to a lower value before running the simulation loop.


In [ ]:
import psutil

# Logical cores (Thread count)
logical = psutil.cpu_count(logical=True)

print(f"Total Threads:  {logical}")

### 5.2 — Helper Functions

The following functions handle all file manipulation and simulation execution steps:

- **`format_radioss_field(value)`** — Formats a numeric value to fit the fixed-width column format required by OpenRadioss input decks.
- **`modify_radioss_starter(...)`** — Reads the master starter file (`_0000.rad`) and writes a modified copy with updated `PROP/SHELL` thicknesses and an optional rigid wall (`RWALL/CYL`) Y-position offset, enabling geometric variation across runs.
- **`copy_engine_file(...)`** — Copies the master engine file (`_0001.rad`) unchanged into each run directory, since solver settings remain constant across all runs.
- **`run_radioss(...)`** — Invokes the `runradioss.bat` script to execute the Starter and Engine sequentially for a given simulation directory.
- **`convert_anim_to_d3plot(...)`** — Converts the ANIM output files produced by the Engine into the d3plot format required by PhysicsNeMo, using the Vortex-Radioss library.


In [ ]:
import sys
import os
import subprocess
import random
from vortex_radioss.animtod3plot.Anim_to_D3plot import readAndConvert


def format_radioss_field(value):
    try:
        formatted = "{:10.2f}".format(value)
        if len(formatted) > 10:
            formatted = "{:.2E}".format(value).replace('E', 'E+').replace('E+-', 'E-')
            if len(formatted) > 10:
                formatted = "{:.1E}".format(value).replace('E', 'E+').replace('E+-', 'E-')

        return formatted.rjust(10)
    except ValueError:
        return "        "

def modify_radioss_starter(input_filepath, output_filepath, prop_changes, pole_y_offset=None):
    """
    Modify multiple PROP/SHELL thicknesses and optionally pole Y position in a RADIOSS starter file.

    Args:
        input_filepath: Path to the input starter file
        output_filepath: Path to the output starter file
        prop_changes: Dictionary mapping prop_id to new_thickness, e.g., {2: 1.5, 3: 2.0}
        pole_y_offset: Optional float to add to the YM and Y_M1 values of /RWALL/CYL/1
    """
    try:
        if not os.path.exists(input_filepath):
            print(f"Error: Input file not found at '{input_filepath}'")
            return

        with open(input_filepath, 'r') as infile, open(output_filepath, 'w') as outfile:

            in_prop_shell = False
            in_rwall = False
            rwall_line_counter = 0
            modifications_made = {}
            prop_id_str = "9999999"
            line_counter = 0
            modified_line = ""

            for line in infile:
                # Check if this line should be skipped (will be replaced with modified version)
                skip_line = False

                # Handle /PROP/SHELL modifications
                if line.strip().startswith('/PROP/SHELL/'):
                    in_prop_shell = True
                    prop_id_str = line[12:].strip()

                current_prop_id = int(prop_id_str)
                if current_prop_id in prop_changes:
                    if line_counter <= 8 and in_prop_shell:
                        line_counter += 1
                        in_prop_shell = True
                    else:
                        line_counter = 0
                        in_prop_shell = False

                    if line_counter == 8:
                        line_counter = 0
                        in_prop_shell = False
                        new_thickness = prop_changes[current_prop_id]
                        modified_line = line[:30] + format_radioss_field(new_thickness) + line[40:]
                        outfile.write(modified_line)
                        modifications_made[current_prop_id] = modifications_made.get(current_prop_id, 0) + 1
                        skip_line = True
                else:
                    if in_prop_shell:
                        line_counter = 0
                        in_prop_shell = False

                # Handle /RWALL/CYL/1 modifications
                if line.strip().startswith('/RWALL/CYL/1'):
                    in_rwall = True
                    rwall_line_counter = 0
                elif in_rwall:
                    rwall_line_counter += 1

                    # Line 7: YM data line (columns 31-40, index 30-40)
                    if rwall_line_counter == 7 and pole_y_offset is not None:
                        # Parse the YM value from columns 31-40
                        ym_str = line[30:40].strip()
                        try:
                            ym_val = float(ym_str)
                            new_ym = ym_val + pole_y_offset
                            modified_line = line[:30] + format_radioss_field(new_ym) + line[40:]
                            outfile.write(modified_line)
                            modifications_made['pole_YM'] = True
                            skip_line = True
                        except ValueError:
                            pass  # Will write original line below
                    # Line 9: Y_M1 data line (columns 31-40, index 30-40)
                    elif rwall_line_counter == 9 and pole_y_offset is not None:
                        # Parse the Y_M1 value from columns 31-40
                        ym1_str = line[30:40].strip()
                        try:
                            ym1_val = float(ym1_str)
                            new_ym1 = ym1_val + pole_y_offset
                            modified_line = line[:30] + format_radioss_field(new_ym1) + line[40:]
                            outfile.write(modified_line)
                            modifications_made['pole_Y_M1'] = True
                            skip_line = True
                        except ValueError:
                            pass  # Will write original line below
                    elif rwall_line_counter >= 10:
                        in_rwall = False
                        rwall_line_counter = 0

                # Write the line if it wasn't skipped
                if not skip_line:
                    outfile.write(line)

            for prop_id in prop_changes:
                if prop_id not in modifications_made:
                    print(f"Warning: /PROP/SHELL card with ID {prop_id} was not found or updated.")

            if pole_y_offset is not None:
                if 'pole_YM' not in modifications_made:
                    print(f"Warning: Pole YM position was not found or updated.")
                if 'pole_Y_M1' not in modifications_made:
                    print(f"Warning: Pole Y_M1 position was not found or updated.")

            print(f"Run saved to: '{output_filepath}'")

    except FileNotFoundError:
        print(f"Error: Could not open one of the files.")
    except Exception as e:
        print(f"Oops! you have to deal with this: {e}")

def copy_engine_file(master_engine_file_with_path, new_engine_file_with_path):

    try:
        if not os.path.exists(master_engine_file_with_path):
            print(f"Error: Input file not found at '{master_engine_file_with_path}'")
            return

        with open(master_engine_file_with_path, 'r') as infile, open(new_engine_file_with_path, 'w') as outfile:

            for line in infile:
                    outfile.write(line)
    except FileNotFoundError:
        print(f"Error: Could not open one of the files.")
    except Exception as e:
        print(f"Oops! you have to deal with this: {e}")

def run_radioss(openradiosspath:str, simpath:str, simfile:str, nt:int):
    # Use generated Windows batch file to run OpenRadioss
    bat_file_directory = "./runradioss.bat"
    # Command to run OpenRadioss
    command = "runradioss.bat" + " " + openradiosspath + " " + simpath + " " + simfile + " " + str(nt)
    # Run the command with "!", to see solver output on the notebook
    !{command}


def convert_anim_to_d3plot(resultfile_dir: str, resultfile: str):
    file_stem = resultfile_dir + "/" + resultfile
    readAndConvert(file_stem, use_shell_mask=False, silent=False)

### 5.3 — Run Simulations

Execute the full batch of simulations. For each run, the script randomly samples shell thickness and pole position values within the defined ranges, modifies the input files accordingly, runs the OpenRadioss solver, and converts the output to d3plot format.

**Configurable parameters:**

| Parameter | Description |
|---|---|
| `number_of_sims` | Total number of simulation variations to generate and run |
| `thickness_range` | Min/max shell thickness values (mm) sampled uniformly at random |
| `pole_y_range` | Min/max Y-axis offset (mm) for the rigid pole impact position |
| `prop_ids_to_change` | List of `PROP/SHELL` IDs in the input deck to update with the sampled thickness |

**Output directory structure:**
```
RAW_DATA/
├── Run100/
│   ├── Bumper_Beam_AP_meshed.d3plot
│   ├── Bumper_Beam_AP_meshed_0000.rad   # Modified starter file
│   └── Bumper_Beam_AP_meshed_0001.rad   # Engine file
├── Run101/
│   └── ...
└── ...
```


In [ ]:
print("GENERATING D3PLOT DATASET FOR PhysicsNemo")
print("       POWERED BY OpenRadioss")
###########################################################
# Number of threads of your processor
nt = logical
sim_path = os.getcwd()
# OpenRadioss path
openradioss_path = sim_path + "/OpenRadioss"
sim_name = "Bumper_Beam_AP_meshed"
# Thickness range (min, max) - same for all props
thickness_range = (1.2, 2.0)
# List of PROP/SHELL IDs to modify with the same thickness
prop_ids_to_change = [2, 7]
# Add more prop IDs as needed
# Pole Y position offset range (min, max)
pole_y_range = (-500, 500)
# Number of simulation runs
number_of_sims = 99
"""""
The generated data structure:

├── Run100/   
│       ├── sim_name.d3plot
│       ├── sim_name_0000.rad     # RADIOSS STARTER FILE
│       └── sim_name_0001.rad     # RADIOSS ENGINE FILE
├── Run101/
│       ├── sim_name.d3plot
│       ├── sim_name_0000.rad
│       └── sim_name_0001.rad
└── ...
"""""
###########################################################

os.mkdir(sim_path + "/RAW_DATA")
for i in range(number_of_sims):
        
    if i < 10:
        run_dir = sim_path + "/RAW_DATA" + "/Run10"+str(i)
    else:
        run_dir = sim_path + "/RAW_DATA" + "/Run1"+str(i)
    os.mkdir(run_dir)
        
    # Generate one random thickness for all props
    new_thickness = random.uniform(thickness_range[0], thickness_range[1])
    prop_changes = {prop_id: new_thickness for prop_id in prop_ids_to_change}
        
# Generate random pole Y offset
pole_y_offset = random.uniform(pole_y_range[0], pole_y_range[1])

master_starter_file = sim_path + "/" + "Bumper_Beam_Radioss_Input" + "/" + sim_name + "_0000.rad"
master_engine_file = sim_path + "/" + "Bumper_Beam_Radioss_Input" + "/" + sim_name + "_0001.rad"

new_starter_file = run_dir + "/" + sim_name + "_0000.rad"
new_engine_file = run_dir + "/" + sim_name + "_0001.rad"


modify_radioss_starter(master_starter_file, new_starter_file, prop_changes, pole_y_offset)
copy_engine_file(master_engine_file, new_engine_file)
        

run_radioss(openradioss_path, run_dir.replace("/", "\\") , sim_name, nt)
convert_anim_to_d3plot(run_dir, sim_name)

## Step 6 — Generate Global Features JSON

Parse all generated starter files (`_0000.rad`) and extract the key simulation parameters into a single `GLOBAL_FEATURES.json` file. This file acts as a lookup table mapping each run to its input conditions, which PhysicsNeMo uses as **global features** during model training.

The following parameters are extracted from each run:

| Field | Source in `_0000.rad` file | Description |
|---|---|---|
| `velocity_x` | `/INIVEL/TRA/1` | Initial impact velocity in the X direction (mm/ms) |
| `rwall_origin_y` | `/RWALL/CYL/1` | Y-coordinate of the rigid pole center (mm) |
| `thickness_scale` | `/PROP/SHELL/2` | Shell thickness applied to the bumper components (mm) |


In [ ]:
import os
import json
from pathlib import Path

def parse_rad_file(filepath):
    """Parses a single 0000.rad file and extracts simulation parameters."""
    data = {}
    try:
        with open(filepath, 'r') as f:
            lines = f.readlines()
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None

    for i, line in enumerate(lines):
        line = line.strip()
        
        # 1. Initial Velocity (velocity_x)
        if line.startswith('/INIVEL/TRA/1'):
            for j in range(i+1, len(lines)):
                if lines[j].startswith('/'): break # Stop if next block starts
                if 'Vx' in lines[j] and 'Vy' in lines[j]:
                    val_line = lines[j+1].split()
                    try:
                        data['velocity_x'] = float(val_line[0])
                    except (IndexError, ValueError):
                        print(f"Warning: Could not parse velocity_x in {filepath}")
                    break
                    
        # 2. Rigid Wall Origin Y (rwall_origin_y)
        elif line.startswith('/RWALL/CYL/1'):
            for j in range(i+1, len(lines)):
                if lines[j].startswith('/'): break # Stop if next block starts
                if 'XM' in lines[j] and 'YM' in lines[j]:
                    val_line = lines[j+1].split()
                    try:
                        # XM is 0, YM is 1
                        data['rwall_origin_y'] = float(val_line[1])
                    except (IndexError, ValueError):
                        print(f"Warning: Could not parse rwall_origin_y in {filepath}")
                    break
                    
        # 3. Shell Thickness (thickness_scale)
        elif line.startswith('/PROP/SHELL/2'):
            for j in range(i+1, len(lines)):
                if lines[j].startswith('/'): break # Stop if next block starts
                if 'Thick' in lines[j]:
                    val_line = lines[j+1].split()
                    try:
                        # N is 0, Istrain is 1, Thick is 2
                        data['thickness_scale'] = float(val_line[2])
                    except (IndexError, ValueError):
                        print(f"Warning: Could not parse thickness_scale in {filepath}")
                    break
    
    # Check for missing keys
    expected_keys = ['velocity_x', 'rwall_origin_y', 'thickness_scale']
    for k in expected_keys:
        if k not in data:
            print(f"Warning: Missing {k} in {filepath}")
            
    return data

def extract_simulation_data(base_dirs, output_file):
    """Iterates over base directories, extracts data from run folders, and saves to JSON."""
    results = {}
    
    for base_dir in base_dirs:
        base_path = Path(base_dir)
        if not base_path.exists():
            print(f"Warning: Directory '{base_dir}' does not exist. Skipping.")
            continue
        
        # Iterate through Run folders (e.g., Run100, Run101)
        for run_dir in base_path.iterdir():
            if run_dir.is_dir():
                # Look for any file ending in 0000.rad
                rad_files = list(run_dir.glob('*0000.rad'))
                if rad_files:
                    rad_file = rad_files[0]
                    parsed_data = parse_rad_file(rad_file)
                    if parsed_data:
                        results[run_dir.name] = parsed_data
                else:
                    print(f"Warning: Missing *0000.rad in '{run_dir}'")
                    
    # Save to JSON
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=4)
    print(f"\nExtraction complete. Data saved to '{output_file}'")

# --- Configuration and Execution ---
if __name__ == '__main__':
    # Set the paths to your directories here
    DIRECTORIES_TO_SCAN = [
        './RAW_DATA'
    ]
    OUTPUT_JSON_FILE = 'GLOBAL_FEATURES.json'
    
    extract_simulation_data(DIRECTORIES_TO_SCAN, OUTPUT_JSON_FILE)